# Calibration, uncertainty, and paired statistical comparisons

**Status:** drafting code for review.

**When to run:** after notebooks 01 and/or 03 have saved row-level test
predictions from fixed, preregistered model configurations.

**Inputs:** `*_test_predictions.csv` files under `tables/drafting_code`.

**Outputs:** optional confidence-interval tables, paired-difference tests,
and calibration figures.

**Usage rule:** compare runs only when complete split hashes, pair identities, row indices and
labels match. Development runs are excluded unless explicitly included.
Bootstrap rows within class to respect imbalance. Treat seeds as repeated
training runs rather than independent biological samples. `QUICK_MODE=True`
uses 500 resamples for review; the reporting run uses 2,000.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and input discovery

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.calibration import calibration_curve

from snooppi_utils import expected_calibration_error, macro_average_precision

WRITE_OUTPUTS = False
RUN_TAG = "calibration_statistics_v1"
QUICK_MODE = True
BOOTSTRAP_REPLICATES = 500 if QUICK_MODE else 2_000
BOOTSTRAP_SEED = 2026
REFERENCE_RUN = None  # Copy an exact run_id displayed below; predeclare before inference.
INCLUDE_DEVELOPMENT_RUNS = False
MAX_CALIBRATION_CURVES = 8  # Explicitly select runs below for larger experiments.
CALIBRATION_RUNS = None
splits = load_splits(PATHS)
assert_model_ready(splits)
manifest = split_manifest(PATHS)
expected_identity = prediction_identity(splits["test"], manifest)

PREDICTION_FILES = None  # Prefer exact filenames for one frozen protocol/run set.
# Example: ["strict_classical_v2_test_predictions.csv", "strict_ablation_v2_test_predictions.csv"]
prediction_files = ([PATHS["tables"] / name for name in PREDICTION_FILES] if PREDICTION_FILES is not None
                    else sorted(PATHS["tables"].glob("*_test_predictions.csv")))
if not prediction_files:
    raise FileNotFoundError(
        f"No test prediction files found in {PATHS['tables']}. "
        "Run notebook 01 or 03 with WRITE_OUTPUTS=True first."
    )
print("Prediction files:", *prediction_files, sep="\n- ")

## 2. Normalize saved prediction schemas

In [ ]:
tables = []
for path in prediction_files:
    frame = pd.read_csv(path)
    required = {"encoder", "seed", "row_index", "label", "probability", "pair_id", "full_pair_id",
                "split_sha256", "split_set_sha256", "threshold", "quick_mode"}
    if required - set(frame):
        raise ValueError(f"{path.name} lacks the revised prediction schema: {sorted(required - set(frame))}")
    frame["source_file"] = path.name
    identity_columns = ["source_file"] + [c for c in ["encoder", "model", "ablation", "seed", "experiment"] if c in frame]
    frame["run_id"] = frame[identity_columns].astype(str).agg("|".join, axis=1)
    for run_id, run in frame.groupby("run_id"):
        run = validate_prediction_identity(run, expected_identity)
        if not INCLUDE_DEVELOPMENT_RUNS and run.quick_mode.iloc[0]:
            print("Excluded development run:", run_id)
            continue
        tables.append(run)
if not tables:
    raise ValueError("No reporting predictions selected. Run 01/03 with QUICK_MODE=False or explicitly enable INCLUDE_DEVELOPMENT_RUNS for smoke review")
predictions = pd.concat(tables, ignore_index=True)
display(predictions.groupby("run_id")["row_index"].nunique().to_frame("test_rows"))


## 3. Stratified bootstrap confidence intervals

In [ ]:
def stratified_bootstrap_indices(labels, replicates, seed):
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(labels == value) for value in (0, 1)]
    for _ in range(replicates):
        sampled = [rng.choice(indices, size=len(indices), replace=True) for indices in class_indices]
        yield np.concatenate(sampled)


interval_rows = []
for run_id, frame in predictions.groupby("run_id", sort=True):
    frame = frame.sort_values("row_index")
    labels = frame["label"].to_numpy(dtype=int)
    scores = frame["probability"].to_numpy(dtype=float)
    estimates = np.array([
        macro_average_precision(labels[index], scores[index])
        for index in stratified_bootstrap_indices(labels, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED)
    ])
    interval_rows.append({
        "run_id": run_id,
        "macro_ap": macro_average_precision(labels, scores),
        "macro_ap_ci_low": np.quantile(estimates, 0.025),
        "macro_ap_ci_high": np.quantile(estimates, 0.975),
        "ece_10_bins": expected_calibration_error(labels, scores, n_bins=10),
        "n": len(labels),
    })

intervals = pd.DataFrame(interval_rows).sort_values("macro_ap")
display(intervals.round(4))

## 4. Paired differences against a declared reference

The same bootstrap rows are applied to both models for the confidence
interval. A paired score-swap randomization test evaluates the equality
null, followed by Holm correction across the displayed comparisons. Keep a
preregistered primary comparison whenever possible.

In [ ]:
def paired_bootstrap_difference(reference, candidate, replicates, seed):
    labels, ref_scores, candidate_scores = align_paired_predictions(reference, candidate)
    differences = []
    for index in stratified_bootstrap_indices(labels, replicates, seed):
        differences.append(
            macro_average_precision(labels[index], candidate_scores[index])
            - macro_average_precision(labels[index], ref_scores[index])
        )
    differences = np.asarray(differences)
    observed = macro_average_precision(labels, candidate_scores) - macro_average_precision(labels, ref_scores)
    # Paired randomization tests the exchangeability null directly: for every
    # test row, randomly swap the two models' scores and recompute the metric.
    rng = np.random.default_rng(seed + 1)
    null_differences = []
    for _ in range(replicates):
        swap = rng.integers(0, 2, size=len(labels), dtype=np.int8).astype(bool)
        permuted_reference = np.where(swap, candidate_scores, ref_scores)
        permuted_candidate = np.where(swap, ref_scores, candidate_scores)
        null_differences.append(
            macro_average_precision(labels, permuted_candidate)
            - macro_average_precision(labels, permuted_reference)
        )
    null_differences = np.asarray(null_differences)
    p_value = (1 + np.sum(np.abs(null_differences) >= abs(observed))) / (replicates + 1)
    return observed, np.quantile(differences, 0.025), np.quantile(differences, 0.975), p_value


def holm_adjust(p_values):
    """Holm family-wise error correction with monotone adjusted p-values."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running_maximum = 0.0
    total = len(p_values)
    for rank, index in enumerate(order):
        running_maximum = max(running_maximum, (total - rank) * p_values[index])
        adjusted[index] = min(1.0, running_maximum)
    return adjusted


available_runs = sorted(predictions["run_id"].unique())
if REFERENCE_RUN is None:
    print("Set REFERENCE_RUN to one exact run_id for paired comparisons.")
    paired_results = pd.DataFrame()
elif REFERENCE_RUN not in available_runs:
    raise ValueError(f"REFERENCE_RUN not found. Available values: {available_runs}")
else:
    reference = predictions.loc[predictions["run_id"].eq(REFERENCE_RUN)]
    paired_rows = []
    for candidate_id in available_runs:
        if candidate_id == REFERENCE_RUN:
            continue
        candidate = predictions.loc[predictions["run_id"].eq(candidate_id)]
        difference, low, high, p_value = paired_bootstrap_difference(
            reference, candidate, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED
        )
        paired_rows.append({
            "reference": REFERENCE_RUN,
            "candidate": candidate_id,
            "macro_ap_difference": difference,
            "ci_low": low,
            "ci_high": high,
            "randomization_p_value": p_value,
        })
    paired_results = pd.DataFrame(paired_rows)
    if not paired_results.empty:
        paired_results = paired_results.sort_values("macro_ap_difference", ascending=False)
        paired_results["holm_adjusted_p_value"] = holm_adjust(paired_results["randomization_p_value"])
        display(paired_results.round(4))

## 5. Calibration curves

In [ ]:
figure, axis = plt.subplots(figsize=(9, 6))
axis.plot([0, 1], [0, 1], color="#333333", linestyle="--", label="Perfect calibration")
selected_runs = CALIBRATION_RUNS or sorted(predictions.run_id.unique())[:MAX_CALIBRATION_CURVES]
print("Displayed calibration runs:", selected_runs)
for run_id, frame in predictions.loc[predictions.run_id.isin(selected_runs)].groupby("run_id"):
    # Quantile bins avoid many empty bins when probabilities are concentrated.
    observed, predicted = calibration_curve(
        frame["label"], frame["probability"], n_bins=10, strategy="quantile"
    )
    axis.plot(predicted, observed, marker="o", linewidth=1.5, label=" | ".join(run_id.split("|")[1:]))
axis.set(
    title="SNOOPPI test-set reliability curves",
    xlabel="Mean predicted interaction score",
    ylabel="Observed positive fraction",
    xlim=(0, 1),
    ylim=(0, 1),
)
axis.legend(frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.18), loc="upper center", ncol=2)
figure.tight_layout()
plt.show()

## 6. Optional outputs

In [ ]:
if WRITE_OUTPUTS:
    output_paths = [
        PATHS["tables"] / f"{RUN_TAG}_bootstrap_intervals.csv",
        PATHS["figures"] / f"{RUN_TAG}_calibration.png",
    ]
    if not paired_results.empty:
        output_paths.append(PATHS["tables"] / f"{RUN_TAG}_paired_differences.csv")
    manifest_path = PATHS["tables"] / f"{RUN_TAG}_manifest.json"
    prepare_outputs(output_paths + [manifest_path])
    save_json(make_run_manifest(PATHS, splits, {"run_tag": RUN_TAG, "quick_mode": QUICK_MODE,
              "bootstrap_replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
              "reference_run": REFERENCE_RUN, "include_development_runs": INCLUDE_DEVELOPMENT_RUNS,
              "prediction_files": {p.name: file_sha256(p) for p in prediction_files}}), manifest_path)
    intervals.to_csv(output_paths[0], index=False)
    figure.savefig(output_paths[1], dpi=300, bbox_inches="tight")
    if not paired_results.empty:
        paired_results.to_csv(output_paths[2], index=False)
    print("Saved statistical outputs.")
else:
    print("Preview only. Set a unique RUN_TAG before saving.")

## Reporting checklist

Report the selected split protocol, class counts/prevalence, all seeds,
point estimates and intervals, the validation-only threshold rule, Brier
score/ECE, and the exact paired comparison. “ProtT5 is best” should be
narrowed to the evaluated protocols and supported by the paired interval,
rather than inferred from overlapping single-model confidence intervals.

Row bootstrap intervals and score-swap tests are exploratory under independent
pair assumptions. Shared proteins/families create dependence even in a strict
partition. Use a reviewed protein/family/component resampling unit for inferential
claims; this notebook does not provide that study-specific clustered analysis.
